# SwinV2-Small — Baseline
Standard CrossEntropyLoss, no imbalance handling. Clean reference point.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))  # so 'from utils import *' works
from utils import *
print("utils loaded ✓")

/scratch/AML_G14_CW/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


utils.py loaded — device=cuda, train=24588, val=5269, test=5269
utils loaded ✓


In [2]:
# ── Output dirs ───────────────────────────────────────────────────────────────
EXP_DIR  = ROOT / 'outputs' / 'swinv2_small_baseline'
CKPT_DIR_B = EXP_DIR / 'checkpoints'
FIG_DIR_B  = EXP_DIR / 'figures'
for d in (CKPT_DIR_B, FIG_DIR_B): d.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = CKPT_DIR_B / 'baseline_best.pt'
print('Output dir:', EXP_DIR)

Output dir: /scratch/AML_G14_CW/outputs/swinv2_small_baseline


In [3]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_ds = DRDataset(train_df, TRAIN_DIR, transform=baseline_train_transform)
val_ds   = DRDataset(val_df,   VAL_DIR,   transform=val_transform)
test_ds  = DRDataset(test_df,  TEST_LABELS_DIR, transform=val_transform)
t_loader = make_loader(train_ds)
v_loader = make_val_loader(val_ds)
test_loader = make_val_loader(test_ds)
print(f'Train: {len(t_loader)} batches | Val: {len(v_loader)} batches')

Train: 1536 batches | Val: 330 batches


In [4]:
# ── Model + loss + optimiser ──────────────────────────────────────────────────
set_seed()
model     = build_swin_model(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = layer_wise_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

Downloaded and loaded.
SwinV2-Small | 49.0M params


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
history, best_kappa = train_loop(
    model, criterion, optimizer, scheduler, scaler,
    t_loader, v_loader, CKPT_PATH)

Epoch       Tr Loss     Tr κ   Val Loss    Val κ
────────────────────────────────────────────────────


1            0.9670   0.3480     0.8965   0.4789  ★


2            0.8866   0.5678     0.8720   0.5805  ★


3            0.8577   0.6151     0.8481   0.6364  ★


4            0.8318   0.6621     0.8401   0.6264


...

7            0.7903   0.7208     0.8301   0.6549


8            0.7787   0.7267     0.8256   0.6651


9            0.7715   0.7407     0.8140   0.6938  ★


10           0.7649   0.7492     0.8140   0.6906


val  :   7%|▋         | 22/330 [00:02<00:34,  8.85it/s]   

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
y_true, y_pred = evaluate(model, criterion, optimizer, scaler,
                           test_loader, CKPT_PATH, 'Baseline')
plot_confusion(y_true, y_pred, 'Baseline', FIG_DIR_B / 'baseline_confusion.png')

In [ ]:
# ── Save results for comparison ───────────────────────────────────────────────
import json as _json
results = {
    'qwk'     : cohen_kappa_score(y_true, y_pred, weights='quadratic'),
    'macro_f1': f1_score(y_true, y_pred, average='macro'),
    'f1_per'  : f1_score(y_true, y_pred, average=None).tolist(),
    'history' : history,
}
with open(EXP_DIR / 'baseline_results.json', 'w') as f:
    _json.dump(results, f, indent=2)
print(f"Test QWK: {results['qwk']:.4f}")
print(f"Macro F1: {results['macro_f1']:.4f}")
print("Results saved — share baseline_results.json with teammates for group comparison table.")